# MAG7 Full Classifier Final Clean

This clean notebook is ordered as the final dissertation pipeline: setup, data loading, pre-specified horizon handling, feature construction, validation-only horizon audit, frozen-rule refit, held-out test evaluation, ablations, robustness checks, and final summary tables.

**CHANGED in clean notebook:** the original notebook has been reordered and lightly restructured so the final `model_df` is built after the horizon audit. The source notebook is left untouched.


## 1. Aim And Setup

This notebook evaluates a full-event MAG7 price-direction classifier. Each retained news event is classified as realised bullish, bearish, or neutral. The terminology uses **training**, **validation**, and **test** throughout.


In [1]:
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import binomtest

import matplotlib.pyplot as plt
import seaborn as sns

import ltn

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

# Use the folder the notebook is currently running from.
WORK_DIR = Path.cwd()

# Search only in the current folder for required input files.
EVENTS_PATH = WORK_DIR / "refined_event_returns.parquet"
MAG7_PRICE_PATH = WORK_DIR / "mag7_daily_prices.parquet"
SP500_PRICE_PATH = WORK_DIR / "sp500_daily_prices.parquet"

required_files = {
    "refined_event_returns.parquet": EVENTS_PATH,
    "mag7_daily_prices.parquet": MAG7_PRICE_PATH,
    "sp500_daily_prices.parquet": SP500_PRICE_PATH,
}

missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required file(s) in the current folder:\n"
        + "\n".join(f"- {name}" for name in missing)
        + f"\n\nCurrent folder is:\n{WORK_DIR}"
    )

# Write all outputs to the current folder.
OUTPUT_DIR = WORK_DIR / "16_mag7_full_classify_final_final_simple_return"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_HORIZON = 3
HORIZON_CANDIDATES = [1, 2, 3, 4, 5]
RETURN_THRESHOLD = 0.005
TRAINING = ("2023-01-01", "2024-12-31")
VALIDATION = ("2025-01-01", "2025-12-31")
TEST = ("2026-01-01", "2026-06-01")

EPOCHS = 500
LR = 0.01
LOGIC_WEIGHT = 0.30
SEED = 7

ROBUST_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

LABELS = ["bearish", "bullish", "neutral"]
ANTECEDENT_THRESHOLD = 0.50
MIN_TRAINING_EVENTS = 6
MIN_VALIDATION_EVENTS = 5
MIN_TRAINING_ACCURACY = 60.0
MIN_VALIDATION_ACCURACY = 60.0

SAVE_OUTPUTS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("working folder:", WORK_DIR)
print("events:", EVENTS_PATH)
print("mag7 prices:", MAG7_PRICE_PATH)
print("sp500 prices:", SP500_PRICE_PATH)
print("output:", OUTPUT_DIR)
print("ltn:", getattr(ltn, "__version__", "unknown"))
print("torch:", torch.__version__, "device:", device)

working folder: /home/jovyan/Stock-Sentiment-Prediction
events: /home/jovyan/Stock-Sentiment-Prediction/refined_event_returns.parquet
mag7 prices: /home/jovyan/Stock-Sentiment-Prediction/mag7_daily_prices.parquet
sp500 prices: /home/jovyan/Stock-Sentiment-Prediction/sp500_daily_prices.parquet
output: /home/jovyan/Stock-Sentiment-Prediction/16_mag7_full_classify_final_final_simple_return
ltn: unknown
torch: 2.12.1+cu126 device: cuda


In [2]:
def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

reset_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not force all deterministic algorithms:", e)

## 2. Load Event And Price Data

This section loads the precomputed MAG7 event table, MAG7 prices, and S&P 500 prices. It defines the return-building helpers but does not yet attach a final return horizon to the modelling frame.


In [3]:
events_all = pd.read_parquet(EVENTS_PATH).copy()
prices = pd.read_parquet(MAG7_PRICE_PATH).copy()
sp500 = pd.read_parquet(SP500_PRICE_PATH).copy()

events_all["time_published"] = pd.to_datetime(events_all["time_published"], utc=True)
events_all["event_date"] = pd.to_datetime(events_all["event_date"])
prices["date"] = pd.to_datetime(prices["date"])
sp500["date"] = pd.to_datetime(sp500["date"])


def build_simple_return_table(price_df: pd.DataFrame, horizons) -> pd.DataFrame:
    frames = []
    for horizon in sorted({int(h) for h in horizons if pd.notna(h)}):
        for ticker, group in price_df.groupby("ticker"):
            g = group.sort_values("date").copy()
            px = pd.to_numeric(g["adj_close"], errors="coerce")
            g["horizon_days"] = horizon
            g["event_date"] = g["date"]
            g["prev_close"] = px.shift(1)
            g["future_close"] = px.shift(-(horizon - 1))
            g["simple_return"] = g["future_close"] / g["prev_close"] - 1.0
            frames.append(g[["ticker", "event_date", "horizon_days", "simple_return", "prev_close", "future_close"]])
    return pd.concat(frames, ignore_index=True)


def realised_label_from_return(value: float, threshold: float = RETURN_THRESHOLD) -> str:
    if pd.isna(value):
        return "neutral"
    if value > threshold:
        return "bullish"
    if value < -threshold:
        return "bearish"
    return "neutral"

# CHANGED in clean notebook:
# Build returns for every candidate horizon before the modelling frame is created.
# The final PRIMARY_HORIZON frame is constructed only after the validation-only horizon audit.
all_return_horizons = sorted(
    set(events_all["horizon_days"].dropna().astype(int).unique())
    | set(HORIZON_CANDIDATES)
    | {PRIMARY_HORIZON}
)
simple_returns = build_simple_return_table(prices, all_return_horizons)

# The event feature frame is horizon-independent. Return columns are added later.
events = (
    events_all
    .drop(columns=["simple_return", "prev_close", "future_close", "target_label"], errors="ignore")
    .drop_duplicates("event_id")
    .sort_values(["ticker", "time_published"])
    .reset_index(drop=True)
)

display(events.groupby(events["event_date"].dt.year).size().rename("events").to_frame())
display(events[[
    "ticker", "time_published", "event_date", "representative_title",
    "expected_label", "event_subtype", "valuation_channel", "materiality_strength", "novelty",
]].head())


,events
event_date,
2023,55
2024,122
2025,140
2026,64


,ticker,time_published,event_date,representative_title,expected_label,event_subtype,valuation_channel,materiality_strength,novelty
0,AAPL,2023-07-03 23:40:57+00:00,2023-07-05,"Settlement checks in the mail from Roundup, Ep...",neutral,lawsuit_settlement,legal_liability,medium,material_update
1,AAPL,2023-07-19 22:53:14+00:00,2023-07-20,Report: Apple built ‘Apple GPT’ chatbot using ...,bullish,product_launch,product_adoption,high,new_event
2,AAPL,2023-08-04 06:26:00+00:00,2023-08-04,"Apple sees sales slump continuing, shares drop...",bearish,demand_decrease,revenue_demand,high,material_update
3,AAPL,2023-08-29 00:00:00+00:00,2023-08-29,"Apellis to lay off 25% of staff, trim research...",bearish,layoffs_cost_cutting,cost_margin,high,material_update
4,AAPL,2023-09-22 00:00:00+00:00,2023-09-22,Calix taps Jabil as part of $42B+ broadband ma...,neutral,partnership_customer_deal,unclear,medium,material_update


## 3. Define Return Horizon Before Modelling

`PRIMARY_HORIZON` is declared in setup, and `HORIZON_CANDIDATES` are available before model training. The final labelled modelling frame is intentionally delayed until after the validation-only horizon audit.


## 4. Build Horizon-Independent Context Features

This section builds market, ticker, news-flow and Qwen-predicate features. These features are constructed before the final return horizon is attached.


In [4]:
def add_price_context(price_df: pd.DataFrame, prefix: str, group_col: str | None = None) -> pd.DataFrame:
    out = price_df.sort_values(([group_col] if group_col else []) + ["date"]).copy()
    grouped = out.groupby(group_col, group_keys=False) if group_col else [(None, out)]
    
    parts = []
    
    for _, g in grouped:
        g = g.sort_values("date").copy()
        px = pd.to_numeric(g["adj_close"], errors="coerce")
        daily = px.pct_change()
        
        for window in [1, 3, 5, 20]:
            g[f"{prefix}_ret_{window}d_pre"] = px.pct_change(window).shift(1)
        
        g[f"{prefix}_abs_ret_1d_pre"] = daily.abs().shift(1)
        g[f"{prefix}_vol_5d_pre"] = daily.rolling(5).std().shift(1)
        g[f"{prefix}_vol_20d_pre"] = daily.rolling(20).std().shift(1)
        
        parts.append(g)
        
    return pd.concat(parts, ignore_index=True)

ticker_ctx = add_price_context(prices, "ticker", "ticker")
sp500_ctx = add_price_context(sp500, "sp500", None).drop(columns=["ticker"], errors="ignore")

ctx = events.merge(
    ticker_ctx[[
        "ticker", "date", "ticker_ret_1d_pre", "ticker_ret_3d_pre", "ticker_ret_5d_pre", "ticker_ret_20d_pre",
        "ticker_abs_ret_1d_pre", "ticker_vol_5d_pre", "ticker_vol_20d_pre",
    ]],
    left_on=["ticker", "event_date"],
    right_on=["ticker", "date"],
    how="left",
).drop(columns=["date"], errors="ignore")

ctx = ctx.merge(
    sp500_ctx[[
        "date", "sp500_ret_1d_pre", "sp500_ret_3d_pre", "sp500_ret_5d_pre", "sp500_ret_20d_pre",
        "sp500_abs_ret_1d_pre", "sp500_vol_5d_pre", "sp500_vol_20d_pre",
    ]],
    left_on="event_date",
    right_on="date",
    how="left",
).drop(columns=["date"], errors="ignore")


for h in ["1d", "3d", "5d", "20d"]:
    ctx[f"relative_ret_{h}_pre"] = ctx[f"ticker_ret_{h}_pre"] - ctx[f"sp500_ret_{h}_pre"]

ctx = ctx.sort_values(["ticker", "time_published"]).reset_index(drop=True)
ctx["event_density_ticker_7d"] = 0.0
ctx["event_density_ticker_30d"] = 0.0
ctx["similar_density_ticker_30d"] = 0.0

for ticker, group in ctx.groupby("ticker"):
    for idx, ts, subtype, channel in zip(
        group.index,
        group["time_published"],
        group["event_subtype"],
        group["valuation_channel"],
    ):
        prev = group.loc[group["time_published"].lt(ts)]
        last7 = prev.loc[prev["time_published"].ge(ts - pd.Timedelta(days=7))]
        last30 = prev.loc[prev["time_published"].ge(ts - pd.Timedelta(days=30))]
        similar30 = last30.loc[
            last30["event_subtype"].eq(subtype)
            | last30["valuation_channel"].eq(channel)
        ]

        ctx.loc[idx, "event_density_ticker_7d"] = len(last7)
        ctx.loc[idx, "event_density_ticker_30d"] = len(last30)
        ctx.loc[idx, "similar_density_ticker_30d"] = len(similar30)


hour = ctx["time_published"].dt.hour + ctx["time_published"].dt.minute / 60
ctx["hour_sin_01"] = (np.sin(2 * np.pi * hour / 24) + 1) / 2
ctx["hour_cos_01"] = (np.cos(2 * np.pi * hour / 24) + 1) / 2
ctx["is_monday"] = (ctx["time_published"].dt.weekday == 0).astype(float)
ctx["is_friday"] = (ctx["time_published"].dt.weekday == 4).astype(float)

display(ctx[[
    "ticker", "event_date", "expected_label",
    "relative_ret_5d_pre", "sp500_vol_5d_pre", "ticker_vol_5d_pre",
    "event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d",
    "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
]].head(10))

,ticker,event_date,expected_label,relative_ret_5d_pre,sp500_vol_5d_pre,ticker_vol_5d_pre,event_density_ticker_7d,event_density_ticker_30d,similar_density_ticker_30d,hour_sin_01,hour_cos_01,is_monday,is_friday
0,AAPL,2023-07-05,neutral,0.009523,0.005807,0.011915,0.0,0.0,0.0,0.456422,0.998097,1.0,0.0
1,AAPL,2023-07-20,bullish,0.007166,0.003793,0.007295,0.0,1.0,0.0,0.355902,0.978786,0.0,0.0
2,AAPL,2023-08-04,bearish,-0.002781,0.008558,0.010984,0.0,1.0,0.0,0.996786,0.443398,0.0,1.0
3,AAPL,2023-08-29,bearish,0.017115,0.009786,0.018306,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0
4,AAPL,2023-09-22,neutral,0.028568,0.007071,0.014146,0.0,1.0,0.0,0.500000,1.000000,0.0,1.0
5,AAPL,2023-10-30,bullish,-0.001674,0.008593,0.013354,0.0,0.0,0.0,0.402455,0.009607,0.0,0.0
6,AAPL,2023-11-14,bearish,0.021998,0.008614,0.012786,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0
7,AAPL,2023-11-30,bullish,-0.009392,0.002293,0.004825,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0
8,AAPL,2023-12-19,bearish,-0.011525,0.005160,0.009768,0.0,1.0,0.0,0.441231,0.996534,1.0,0.0
9,AAPL,2024-01-10,bearish,-0.005576,0.008351,0.014335,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0


## 5. Create Chronological Training / Validation / Test Split

This section creates the chronological split and feature specifications. It builds `feature_df`, which is horizon-independent: it contains features and period labels, but not the final `simple_return` or `target_label`.


In [5]:
def assign_period(d):
    d = pd.Timestamp(d)
    if pd.Timestamp(TRAINING[0]) <= d <= pd.Timestamp(TRAINING[1]):
        return "training"
    if pd.Timestamp(VALIDATION[0]) <= d <= pd.Timestamp(VALIDATION[1]):
        return "validation"
    if pd.Timestamp(TEST[0]) <= d <= pd.Timestamp(TEST[1]):
        return "test"
    return "outside"

ctx["period"] = ctx["event_date"].map(assign_period)

fit_mask = ctx["period"].eq("training")

def fit_abs_scale(col, q=0.80):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    scale = s.abs().quantile(q)
    return float(scale) if np.isfinite(scale) and scale != 0 else 1.0

def fit_quantile(col, q, fallback=1.0):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    value = s.quantile(q)
    return float(value) if np.isfinite(value) and value != 0 else fallback

def soft_pos_with_scale(s, scale):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s.clip(lower=0) / scale).clip(0, 1).fillna(0)

def soft_neg_with_scale(s):
    return soft_pos_with_scale(-pd.to_numeric(s, errors="coerce"), scale)

def soft_neg_with_scale(s, scale):
    return soft_pos_with_scale(-pd.to_numeric(s, errors="coerce"), scale)

def soft_low_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (1 - (s / cutoff)).clip(0, 1).fillna(0)

def soft_high_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s / cutoff - 1).clip(0, 1).fillna(0)

ticker_5d_scale = fit_abs_scale("ticker_ret_5d_pre")
market_5d_scale = fit_abs_scale("sp500_ret_5d_pre")
relative_5d_scale = fit_abs_scale("relative_ret_5d_pre")

ctx["ticker_prior_up_5d"] = soft_pos_with_scale(ctx["ticker_ret_5d_pre"], ticker_5d_scale)
ctx["ticker_prior_down_5d"] = soft_neg_with_scale(ctx["ticker_ret_5d_pre"], ticker_5d_scale)
ctx["market_prior_up_5d"] = soft_pos_with_scale(ctx["sp500_ret_5d_pre"], market_5d_scale)
ctx["market_prior_down_5d"] = soft_neg_with_scale(ctx["sp500_ret_5d_pre"], market_5d_scale)
ctx["relative_prior_up_5d"] = soft_pos_with_scale(ctx["relative_ret_5d_pre"], relative_5d_scale)
ctx["relative_prior_down_5d"] = soft_neg_with_scale(ctx["relative_ret_5d_pre"], relative_5d_scale)

ctx["quiet_market_5d"] = soft_low_with_cutoff(
    ctx["sp500_vol_5d_pre"],
    fit_quantile("sp500_vol_5d_pre", 0.40),
)
ctx["volatile_market_5d"] = soft_high_with_cutoff(
    ctx["sp500_vol_5d_pre"],
    fit_quantile("sp500_vol_5d_pre", 0.70),
)
ctx["quiet_ticker_5d"] = soft_low_with_cutoff(
    ctx["ticker_vol_5d_pre"],
    fit_quantile("ticker_vol_5d_pre", 0.40),
)
ctx["volatile_ticker_5d"] = soft_high_with_cutoff(
    ctx["ticker_vol_5d_pre"],
    fit_quantile("ticker_vol_5d_pre", 0.70),
)

for col in ["event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d"]:
    denom = max(ctx.loc[fit_mask, col].quantile(0.95), 1)
    ctx[col + "_01"] = (ctx[col] / denom).clip(0, 1)

ctx["novel_context_30d"] = (1 - ctx["similar_density_ticker_30d_01"]).clip(0, 1)

dummy_parts = []
dummy_cols = []

categorical_cols = [
    "expected_label",
    "event_subtype",
    "valuation_channel",
    "materiality_strength",
    "novelty",
]

for col in categorical_cols:
    categories = sorted(set(ctx.loc[fit_mask, col].fillna("missing").astype(str)))
    values = ctx[col].fillna("missing").astype(str)

    for category in categories:
        dummy_name = f"{col}_{category}"
        dummy_parts.append(values.eq(category).astype(float).rename(dummy_name))
        dummy_cols.append(dummy_name)

dummies = pd.concat(dummy_parts, axis=1)
ctx = pd.concat([ctx, dummies], axis=1)

context_cols = [
    "ticker_prior_up_5d", "ticker_prior_down_5d", "market_prior_up_5d", "market_prior_down_5d",
    "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
    "quiet_ticker_5d", "volatile_ticker_5d",
    "event_density_ticker_7d_01", "event_density_ticker_30d_01", "similar_density_ticker_30d_01",
    "novel_context_30d", "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
]

qwen_cols = list(dummies.columns) + ["include_candidate", "article_count"]
for col in ["include_candidate", "article_count"]:
    ctx[col] = pd.to_numeric(ctx[col], errors="coerce").fillna(0)

article_count_denom = max(ctx.loc[fit_mask, "article_count"].quantile(0.95), 1)
ctx["article_count"] = (ctx["article_count"] / article_count_denom).clip(0, 1)

# Interaction features: Qwen event predicates multiplied by context truth values.
interaction_qwen_cols = [
    c for c in dummy_cols
    if c.startswith("expected_label_")
    or c.startswith("valuation_channel_")
    or c.startswith("event_subtype_")
    or c.startswith("materiality_strength_")
]

interaction_context_cols = [
    "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
    "quiet_ticker_5d", "volatile_ticker_5d", "event_density_ticker_7d_01", "novel_context_30d",
]

interaction_data = {}
for q in interaction_qwen_cols:
    for c in interaction_context_cols:
        name = f"interaction__{q}__x__{c}"
        interaction_data[name] = ctx[q].fillna(0) * ctx[c].fillna(0)

interaction_cols = list(interaction_data.keys())
if interaction_data:
    ctx = pd.concat([ctx, pd.DataFrame(interaction_data, index=ctx.index)], axis=1)

# CHANGED in clean notebook:
# This frame contains event predicates, context features, interactions, and the chronological split only.
# Future returns and realised labels are merged in after the horizon audit.
feature_df = ctx.loc[ctx["period"].isin(["training", "validation", "test"])].copy()

for col in qwen_cols + context_cols + interaction_cols:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

rule_block_feature_cols = {
    "qwen_semantic": list(qwen_cols),
    "market_context": list(context_cols),
    "qwen_context_interaction": list(interaction_cols),
}

display(feature_df["period"].value_counts().rename("events").to_frame())

variant_specs = {
    "full_ltn_qwen_only": {
        "feature_cols": qwen_cols,
        "rule_blocks": ["qwen_semantic"],
    },
    "full_ltn_context_only": {
        "feature_cols": context_cols,
        "rule_blocks": ["market_context"],
    },
    "full_ltn_qwen_plus_context": {
        "feature_cols": qwen_cols + context_cols,
        "rule_blocks": ["qwen_semantic", "market_context"],
    },
    "full_ltn_qwen_context_interaction": {
        "feature_cols": qwen_cols + context_cols + interaction_cols,
        "rule_blocks": ["qwen_semantic", "market_context", "qwen_context_interaction"],
    },
}

for spec in variant_specs.values():
    spec["feature_cols"] = list(dict.fromkeys(spec["feature_cols"]))
    spec["return_col"] = "simple_return"

feature_manifest = pd.DataFrame([
    {
        "variant": name,
        "n_features": len(spec["feature_cols"]),
        "rule_blocks": ", ".join(spec["rule_blocks"]),
        "features": ", ".join(spec["feature_cols"]),
    }
    for name, spec in variant_specs.items()
])

if SAVE_OUTPUTS:
    feature_manifest.to_csv(OUTPUT_DIR / "full_ltn_feature_manifest.csv", index=False)

display(feature_manifest)

,events
period,
training,177
validation,140
test,64


,variant,n_features,rule_blocks,features
0,full_ltn_qwen_only,46,qwen_semantic,"expected_label_bearish, expected_label_bullish..."
1,full_ltn_context_only,18,market_context,"ticker_prior_up_5d, ticker_prior_down_5d, mark..."
2,full_ltn_qwen_plus_context,64,"qwen_semantic, market_context","expected_label_bearish, expected_label_bullish..."
3,full_ltn_qwen_context_interaction,384,"qwen_semantic, market_context, qwen_context_in...","expected_label_bearish, expected_label_bullish..."


## 6. Define Model Variants And LTN Helpers

This section defines the LTN predicates, rule-mining utilities, model-training routine, scoring helper, and summary metrics.


In [6]:
class OutcomeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = max(12, min(80, input_dim * 2))
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).reshape(-1)

def truth_value(obj):
    return obj.value if hasattr(obj, "value") else obj

def feature_tensor(data, cols):
    x = data[cols].replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1).to_numpy(dtype=np.float32)
    return torch.tensor(x, dtype=torch.float32, device=device)

def target_tensors(data):
    y = pd.get_dummies(data["target_label"]).reindex(columns=LABELS, fill_value=0).to_numpy(dtype=np.float32)
    return {
        "bearish": torch.tensor(y[:, 0].reshape(-1, 1), dtype=torch.float32, device=device),
        "bullish": torch.tensor(y[:, 1].reshape(-1, 1), dtype=torch.float32, device=device),
        "neutral": torch.tensor(y[:, 2].reshape(-1, 1), dtype=torch.float32, device=device),
    }

def label_return_alignment(data, label, return_col):
    ret = pd.to_numeric(data[return_col], errors="coerce")
    if label == "bullish":
        signed = ret
    elif label == "bearish":
        signed = -ret
    else:
        signed = -ret.abs()
    signed = signed.dropna()
    if len(signed) < 3:
        return np.nan, np.nan
    if label == "neutral":
        # For neutral, higher is better when absolute returns are smaller.
        p = np.nan
    else:
        p = binomtest(int((signed > 0).sum()), len(signed), 0.5, alternative="greater").pvalue
    return float(100 * signed.mean()), float(p) if np.isfinite(p) else np.nan

def empirical_rule_stats(data, condition_cols, label, return_col):
    if len(data) == 0:
        return {"n": 0, "accuracy_pct": np.nan, "aligned_mean_pct": np.nan, "aligned_signflip_p": np.nan}
    active = np.ones(len(data), dtype=bool)
    for col in condition_cols:
        active &= pd.to_numeric(data[col], errors="coerce").fillna(0).to_numpy() >= ANTECEDENT_THRESHOLD
    subset = data.loc[active].copy()
    if subset.empty:
        return {"n": 0, "accuracy_pct": np.nan, "aligned_mean_pct": np.nan, "aligned_signflip_p": np.nan}
    aligned_mean, p = label_return_alignment(subset, label, return_col)
    return {
        "n": int(len(subset)),
        "accuracy_pct": float(100 * subset["target_label"].eq(label).mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": p,
    }

def infer_rule_block(feature_name):
    if feature_name.startswith("gate__"):
        return "context_gated_qwen"

    if feature_name in context_cols:
        return "market_context"

    if feature_name in qwen_cols:
        return "qwen_semantic"

    return "unknown"

def mine_validated_rules(
    data,
    feature_cols,
    return_col,
    rule_blocks=None,
    max_pair_features=36,
    max_rules_per_label=12,
):
    training_df = data.loc[data["period"].eq("training")].copy()
    validation_df = data.loc[data["period"].eq("validation")].copy()

    cols = [c for c in feature_cols if c in data.columns]

    if rule_blocks is not None:
        allowed_blocks = set(rule_blocks)
        cols = [c for c in cols if infer_rule_block(c) in allowed_blocks]

    active_counts = training_df[cols].ge(ANTECEDENT_THRESHOLD).sum().sort_values(ascending=False)
    pair_base = active_counts.head(max_pair_features).index.tolist()

    candidates = [(c,) for c in cols] + list(combinations(pair_base, 2))

    rows = []

    for condition_cols in candidates:
        condition_cols = tuple(condition_cols)

        for label in LABELS:
            disc = empirical_rule_stats(training_df, condition_cols, label, return_col)

            if disc["n"] < MIN_TRAINING_EVENTS or not np.isfinite(disc["accuracy_pct"]):
                continue

            val = empirical_rule_stats(validation_df, condition_cols, label, return_col)

            if val["n"] < MIN_VALIDATION_EVENTS or not np.isfinite(val["accuracy_pct"]):
                continue

            score = (
                disc["accuracy_pct"] / 100
                + val["accuracy_pct"] / 100
                + min(disc["n"] / 40, 1)
                + min(val["n"] / 30, 1)
            )

            if label != "neutral" and np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(disc["aligned_mean_pct"], 0) / 2, 1)

            if label == "neutral" and np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(-disc["aligned_mean_pct"], 0) / 1, 1)

            rows.append({
                "condition": " & ".join(condition_cols),
                "condition_cols_json": json.dumps(condition_cols),
                "rule_label": label,
                "rule_score": score,
                # CHANGED in clean notebook:
                # Store which rule blocks are involved, so the selected rules are auditable.
                "rule_blocks": ", ".join(sorted({infer_rule_block(c) for c in condition_cols})),
                **{f"training_{k}": v for k, v in disc.items()},
                **{f"validation_{k}": v for k, v in val.items()},
            })

    candidates_df = pd.DataFrame(rows)

    if candidates_df.empty:
        return candidates_df, candidates_df

    selected = candidates_df.loc[
        candidates_df["training_accuracy_pct"].ge(MIN_TRAINING_ACCURACY)
        & candidates_df["validation_accuracy_pct"].ge(MIN_VALIDATION_ACCURACY)
    ].copy()

    selected = selected.sort_values(
        ["rule_label", "validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=[True, False, False, False],
    )

    selected = selected.groupby("rule_label", group_keys=False).head(max_rules_per_label)

    selected = selected.sort_values(
        ["validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=False,
    ).reset_index(drop=True)

    return (
        candidates_df.sort_values("rule_score", ascending=False).reset_index(drop=True),
        selected,
    )

class FeaturePredicate(nn.Module):
    def __init__(self, index):
        super().__init__()
        self.index = index

    def forward(self, z):
        return z[:, self.index].reshape(-1, 1)

def make_feature_predicates(feature_cols):
    preds = {}
    for i, col in enumerate(feature_cols):
        preds[col] = ltn.Predicate(model=FeaturePredicate(i).to(device))
    return preds

def antecedent_from_rule(cols, feat_preds, x, AndOp):
    expr = truth_value(feat_preds[cols[0]](x))
    for col in cols[1:]:
        expr = AndOp(expr, truth_value(feat_preds[col](x)))
    return expr

def train_variant(name, spec, train_df, use_logic=True, logic_weight=LOGIC_WEIGHT, seed=SEED, mine_rules=True):
    reset_seed(seed)
    
    feature_cols = spec["feature_cols"]
    return_col = spec["return_col"]

    # mine fresh rules (training)
    if use_logic and mine_rules:
        candidates, selected_rules = mine_validated_rules(
            train_df,
            feature_cols,
            return_col,
            rule_blocks=spec.get("rule_blocks"),
            max_pair_features=spec.get("max_pair_features", 36),
            max_rules_per_label=spec.get("max_rules_per_label", 12),
        )

    # use frozen rules (after validation)
    elif use_logic and not mine_rules:
        candidates = pd.DataFrame()

        if "selected_rules" not in spec:
            raise ValueError(
                f"{name} was called with mine_rules=False, "
                "but spec['selected_rules'] was not provided."
            )

        selected_rules = spec["selected_rules"].copy()
        
    else:
        candidates = pd.DataFrame()
        selected_rules = pd.DataFrame()
    
    if SAVE_OUTPUTS and use_logic:
        candidates.to_csv(OUTPUT_DIR / f"{name}_candidate_mined_rules.csv", index=False)
        selected_rules.to_csv(OUTPUT_DIR / f"{name}_selected_validated_rules.csv", index=False)

    X = feature_tensor(train_df, feature_cols)
    x = ltn.Variable(f"x_{name}", X)
    y = target_tensors(train_df)
    feat_preds = make_feature_predicates(feature_cols)

    Bearish = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))
    Bullish = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))
    Neutral = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))

    AndOp = ltn.fuzzy_ops.AndProd()
    ImpliesOp = ltn.fuzzy_ops.ImpliesReichenbach()
    ForallAgg = ltn.fuzzy_ops.AggregPMeanError(p=2)
    SatAgg = ltn.fuzzy_ops.SatAgg()

    label_predicates = {"bearish": Bearish, "bullish": Bullish, "neutral": Neutral}
    
    rule_specs = []
    for _, rule in selected_rules.iterrows():
        cols = [c for c in json.loads(rule["condition_cols_json"]) if c in feat_preds]
        if cols:
            rule_specs.append((rule["condition"], cols, rule["rule_label"]))

    params = (
        list(Bearish.model.parameters())
        + list(Bullish.model.parameters())
        + list(Neutral.model.parameters())
    )

    opt = torch.optim.Adam(params, lr=LR)
    
    history = []
    for epoch in range(EPOCHS):
        opt.zero_grad()
        
        pred_bear = truth_value(Bearish(x)).reshape(-1, 1)
        pred_bull = truth_value(Bullish(x)).reshape(-1, 1)
        pred_neut = truth_value(Neutral(x)).reshape(-1, 1)
        
        supervised = (
            F.mse_loss(pred_bear, y["bearish"])
            + F.mse_loss(pred_bull, y["bullish"])
            + F.mse_loss(pred_neut, y["neutral"])
        )
        
        exclusivity = torch.mean(
            pred_bear * pred_bull
            + pred_bear * pred_neut
            + pred_bull * pred_neut
        )

        
        formulas = []
        if use_logic:            
            for rule_name, cols, label in rule_specs:
                formulas.append((
                    rule_name,
                    ForallAgg(
                        ImpliesOp(
                            antecedent_from_rule(cols, feat_preds, x, AndOp),
                            truth_value(label_predicates[label](x)),
                        )
                    ),
                ))
                
        sat = SatAgg(*[formula for _, formula in formulas]) if formulas else torch.tensor(1.0, device=device)
        
        logic_penalty = logic_weight * (1.0 - truth_value(sat)) if use_logic else torch.tensor(0.0, device=device)
        
        loss = supervised + 0.10 * exclusivity + logic_penalty
        loss.backward()
        opt.step()
        
        if epoch % 50 == 0 or epoch == EPOCHS - 1:
            row = {
                "variant": name,
                "epoch": epoch,
                "seed": seed,
                "use_logic": use_logic,
                "mine_rules": mine_rules,
                "logic_weight": logic_weight,
                "loss": float(loss.detach().cpu()),
                "supervised_mse": float(supervised.detach().cpu()),
                "exclusivity": float(exclusivity.detach().cpu()),
                "sat": float(truth_value(sat).detach().cpu()) if hasattr(truth_value(sat), "detach") else float(sat),
                "n_selected_rules": len(rule_specs),
            }
            
            for rule_name, formula in formulas[:20]:
                row[f"sat_{rule_name[:80]}"] = float(truth_value(formula).detach().cpu())
                
            history.append(row)
    return {
        "name": name,
        "seed": seed,
        "feature_cols": feature_cols,
        "candidate_rules": candidates,
        "selected_rules": selected_rules,
        "use_logic": use_logic,
        "mine_rules": mine_rules,
        "logic_weight": logic_weight,
        "Bearish": Bearish,
        "Bullish": Bullish,
        "Neutral": Neutral,
        "history": pd.DataFrame(history),
    }


In [7]:
def score_variant(bundle, df):
    out = df.copy()
    X = feature_tensor(out, bundle["feature_cols"])

    with torch.no_grad():
        eval_x = ltn.Variable(f"eval_{bundle['name']}", X)
        out["score_bearish"] = truth_value(bundle["Bearish"](eval_x)).detach().cpu().numpy().reshape(-1)
        out["score_bullish"] = truth_value(bundle["Bullish"](eval_x)).detach().cpu().numpy().reshape(-1)
        out["score_neutral"] = truth_value(bundle["Neutral"](eval_x)).detach().cpu().numpy().reshape(-1)

    scores = out[["score_bearish", "score_bullish", "score_neutral"]].to_numpy()
    out["prediction"] = np.array(LABELS)[scores.argmax(axis=1)]
    out["confidence_margin"] = np.sort(scores, axis=1)[:, -1] - np.sort(scores, axis=1)[:, -2]

    return out

def aligned_stats(df):
    ret = pd.to_numeric(df["simple_return"], errors="coerce")
    pred = df["prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )

    signed = pd.Series(signed, index=df.index).dropna()

    if len(signed) < 3:
        return np.nan, np.nan

    p = binomtest(int((signed > 0).sum()), len(signed), 0.5, alternative="greater").pvalue

    return float(signed.mean() * 100), float(p)

def summary_row(variant, period, df, bundle=None):
    aligned_mean, aligned_p = aligned_stats(df)

    row = {
        "variant": variant,
        "period": period,
        "n": int(len(df)),
        "accuracy_pct": float(100 * df["prediction"].eq(df["target_label"]).mean()),
        "mean_simple_return_pct": float(100 * pd.to_numeric(df["simple_return"], errors="coerce").mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": aligned_p,
        "bullish_pred_rate_pct": float(100 * df["prediction"].eq("bullish").mean()),
        "bearish_pred_rate_pct": float(100 * df["prediction"].eq("bearish").mean()),
        "neutral_pred_rate_pct": float(100 * df["prediction"].eq("neutral").mean()),
        "mean_confidence_margin": float(df["confidence_margin"].mean()),
    }

    if bundle is not None:
        row["use_logic"] = bundle["use_logic"]
        row["logic_weight"] = bundle["logic_weight"]
        row["n_selected_rules"] = len(bundle["selected_rules"])

    return row


## 7. Validation-Only Horizon Audit

This audit compares candidate return horizons using training and validation data only. The held-out test period is excluded from this step.

**CHANGED in clean notebook:** the audit now builds each horizon-specific modelling frame from `feature_df`, so the final `PRIMARY_HORIZON` frame is not created until after this audit.


In [8]:
# CHANGED in clean notebook:
# The horizon audit now runs before the final PRIMARY_HORIZON model_df is constructed.
# It uses the horizon-independent feature_df and merges each candidate horizon's returns inside the loop.
# Training + validation only are used here; the held-out test period is not used for horizon choice.

HORIZON_AUDIT_VARIANT = "full_ltn_qwen_plus_context"


def build_model_df_for_horizon(horizon: int) -> pd.DataFrame:
    horizon_returns = simple_returns.loc[
        simple_returns["horizon_days"].eq(int(horizon)),
        ["ticker", "event_date", "simple_return", "prev_close", "future_close"],
    ]

    # Adding .copy() at the end of the merge chain consolidates 
    # the fragmented block manager into a clean, single memory block.
    out = (
        feature_df
        .drop(columns=["simple_return", "prev_close", "future_close", "target_label"], errors="ignore")
        .merge(
            horizon_returns,
            on=["ticker", "event_date"],
            how="left",
        )
    ).copy() 

    missing_returns = int(out["simple_return"].isna().sum())
    if missing_returns:
        print(f"{horizon}d horizon: dropping {missing_returns} rows with missing future returns")
        out = out.loc[out["simple_return"].notna()].copy()

    out["target_label"] = out["simple_return"].map(
        lambda x: realised_label_from_return(float(x), RETURN_THRESHOLD)
    )

    return out


horizon_ltn_rows = []

for horizon in HORIZON_CANDIDATES:
    print(f"LTN horizon audit: {horizon}-day simple return")

    model_df_h = build_model_df_for_horizon(horizon)

    train_h = model_df_h.loc[model_df_h["period"].eq("training")].copy()
    validation_h = model_df_h.loc[model_df_h["period"].eq("validation")].copy()
    train_val_h = model_df_h.loc[
        model_df_h["period"].isin(["training", "validation"])
    ].copy()

    spec_base_h = variant_specs[HORIZON_AUDIT_VARIANT].copy()

    candidate_rules_h, selected_rules_h = mine_validated_rules(
        train_val_h,
        spec_base_h["feature_cols"],
        spec_base_h["return_col"],
        rule_blocks=spec_base_h.get("rule_blocks"),
        max_pair_features=spec_base_h.get("max_pair_features", 36),
        max_rules_per_label=spec_base_h.get("max_rules_per_label", 12),
    )

    spec_h = spec_base_h.copy()
    spec_h["selected_rules"] = selected_rules_h.copy()

    bundle_h = train_variant(
        f"{HORIZON_AUDIT_VARIANT}_horizon_{horizon}d",
        spec_h,
        train_h,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

    scored_validation_h = score_variant(bundle_h, validation_h)

    row_h = summary_row(
        f"LTN horizon {horizon}d",
        "validation",
        scored_validation_h,
        bundle=bundle_h,
    )

    majority_baseline_h = validation_h["target_label"].value_counts(normalize=True).max() * 100

    row_h["horizon_days"] = horizon
    row_h["training_n"] = len(train_h)
    row_h["validation_n"] = len(validation_h)
    row_h["validation_majority_baseline_pct"] = float(majority_baseline_h)
    row_h["validation_bullish_n"] = int(validation_h["target_label"].eq("bullish").sum())
    row_h["validation_bearish_n"] = int(validation_h["target_label"].eq("bearish").sum())
    row_h["validation_neutral_n"] = int(validation_h["target_label"].eq("neutral").sum())
    row_h["n_candidate_rules"] = len(candidate_rules_h)
    row_h["n_selected_rules"] = len(selected_rules_h)

    horizon_ltn_rows.append(row_h)


horizon_ltn_audit = pd.DataFrame(horizon_ltn_rows)

display(
    horizon_ltn_audit.sort_values(
        ["accuracy_pct", "aligned_mean_pct", "n_selected_rules"],
        ascending=False,
    )
)

LTN horizon audit: 1-day simple return
LTN horizon audit: 2-day simple return
LTN horizon audit: 3-day simple return
3d horizon: dropping 1 rows with missing future returns
LTN horizon audit: 4-day simple return
4d horizon: dropping 1 rows with missing future returns
LTN horizon audit: 5-day simple return
5d horizon: dropping 2 rows with missing future returns


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules,horizon_days,training_n,validation_n,validation_majority_baseline_pct,validation_bullish_n,validation_bearish_n,validation_neutral_n,n_candidate_rules
3,LTN horizon 4d,validation,140,45.000000,0.335631,0.632597,0.533657,67.142857,32.857143,0.000000,0.675460,True,0.3,17,4,177,140,49.285714,69,57,14,1203
1,LTN horizon 2d,validation,140,38.571429,0.211167,0.059972,0.570940,52.142857,37.142857,10.714286,0.656803,True,0.3,13,2,177,140,46.428571,65,53,22,1203
2,LTN horizon 3d,validation,140,37.857143,0.497367,-0.227043,0.638606,45.000000,45.714286,9.285714,0.674057,True,0.3,10,3,177,140,49.285714,69,51,20,1203
4,LTN horizon 5d,validation,140,37.142857,0.251408,-0.368102,0.882534,70.714286,28.571429,0.714286,0.753900,True,0.3,12,5,177,140,45.000000,60,63,17,1203
0,LTN horizon 1d,validation,140,35.714286,0.242302,-0.186945,0.789238,31.428571,39.285714,29.285714,0.615000,True,0.3,4,1,177,140,47.857143,67,48,25,1203


### 8. Select Primary Horizon And Build Final Model Frame

After the horizon audit, the configured `PRIMARY_HORIZON` is attached to the feature frame to create the final `model_df`. This is the frame used for frozen-rule refitting and held-out testing.


In [9]:
# based on validation accuracy, update returns horizon to 4 days.
PRIMARY_HORIZON = 4

In [10]:
# CHANGED in clean notebook:
# Construct the final modelling frame only after the validation-only horizon audit.
# PRIMARY_HORIZON remains the configured retained horizon for the final held-out evaluation.

model_df = build_model_df_for_horizon(PRIMARY_HORIZON)

train = model_df.loc[model_df["period"].eq("training")].copy()
validation = model_df.loc[model_df["period"].eq("validation")].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

display(model_df.groupby("period")["target_label"].value_counts().unstack(fill_value=0))


4d horizon: dropping 1 rows with missing future returns


target_label,bearish,bullish,neutral
period,,,
test,33,25,5
training,76,85,16
validation,57,69,14


## 9. Mine/Select Rules, Freeze Them, Then Refit

Candidate rules are selected using training plus validation only. The selected rules are frozen before the final model is refit on training plus validation.


### LTN variants experiment 1

This experiment compares LTN feature variants after the primary horizon has been selected. Rules are selected using training and validation data, then frozen. Each variant is refit on training plus validation and evaluated once on the held-out test set.

In [11]:
train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()

frozen_variant_specs = {}

for name, spec in variant_specs.items():
    print("selecting frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_variant_specs[name] = frozen_spec


trained_final = {}

for name, spec in frozen_variant_specs.items():
    print("final frozen-rule training", name, "rows", len(train_val))

    trained_final[name] = train_variant(
        name,
        spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )


final_rows = []
final_frames = []

for variant, bundle in trained_final.items():
    scored = score_variant(bundle, test)

    final_rows.append(
        summary_row(
            variant,
            "test_frozen_rules_refit_training_validation",
            scored,
            bundle=bundle,
        )
    )

    final_frames.append(
        scored.assign(
            variant=variant,
            period="test_frozen_rules_refit_training_validation",
        )
    )


final_summary = pd.DataFrame(final_rows)

if SAVE_OUTPUTS:
    final_summary.to_csv(
        OUTPUT_DIR / "full_ltn_final_frozen_rule_test_summary.csv",
        index=False,
    )

    pd.concat(final_frames, ignore_index=True).to_csv(
        OUTPUT_DIR / "full_ltn_final_frozen_rule_test_predictions.csv",
        index=False,
    )


display(
    final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False,
    )
)

selecting frozen rules full_ltn_qwen_only
selecting frozen rules full_ltn_context_only
selecting frozen rules full_ltn_qwen_plus_context
selecting frozen rules full_ltn_qwen_context_interaction
final frozen-rule training full_ltn_qwen_only rows 317
final frozen-rule training full_ltn_context_only rows 317
final frozen-rule training full_ltn_qwen_plus_context rows 317
final frozen-rule training full_ltn_qwen_context_interaction rows 317


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
3,full_ltn_qwen_context_interaction,test_frozen_rules_refit_training_validation,63,55.555556,0.337596,0.479178,0.064959,50.793651,49.206349,0.0,0.718791,True,0.3,17
0,full_ltn_qwen_only,test_frozen_rules_refit_training_validation,63,53.968254,0.337596,0.716406,0.103684,61.904762,38.095238,0.0,0.540365,True,0.3,13
2,full_ltn_qwen_plus_context,test_frozen_rules_refit_training_validation,63,53.968254,0.337596,0.493966,0.064959,50.793651,49.206349,0.0,0.774148,True,0.3,17
1,full_ltn_context_only,test_frozen_rules_refit_training_validation,63,42.857143,0.337596,-0.492540,0.775019,52.380952,47.619048,0.0,0.629908,True,0.3,3


## 10. LTN variants experiment 2 - test on longer window

In [12]:
# Robustness check: longer held-out test window
# Training: existing early period
# Validation: first half of 2025
# Test: 2025-07-01 to 2026-06-01
#
# This preserves temporal ordering:
# training -> validation -> test

ROBUST_TRAINING = ("2023-06-01", "2024-12-31")
ROBUST_VALIDATION = ("2025-01-01", "2025-06-30")
ROBUST_TEST = ("2025-07-01", "2026-06-01")

def assign_robust_periods(df, date_col="event_date"):
    out = df.copy()
    d = pd.to_datetime(out[date_col]).dt.tz_localize(None)

    out["period_robust"] = np.select(
        [
            d.between(pd.Timestamp(ROBUST_TRAINING[0]), pd.Timestamp(ROBUST_TRAINING[1]), inclusive="both"),
            d.between(pd.Timestamp(ROBUST_VALIDATION[0]), pd.Timestamp(ROBUST_VALIDATION[1]), inclusive="both"),
            d.between(pd.Timestamp(ROBUST_TEST[0]), pd.Timestamp(ROBUST_TEST[1]), inclusive="both"),
        ],
        ["training", "validation", "test"],
        default="outside",
    )
    return out

model_df_robust = assign_robust_periods(model_df, "event_date")

model_df_robust = model_df_robust.loc[
    model_df_robust["period_robust"].isin(["training", "validation", "test"])
].copy()

model_df_robust["period_original"] = model_df_robust["period"]
model_df_robust["period"] = model_df_robust["period_robust"]

print("Robustness split counts:")
display(model_df_robust.groupby("period")["target_label"].value_counts().unstack(fill_value=0))

train_val_robust = model_df_robust.loc[
    model_df_robust["period"].isin(["training", "validation"])
].copy()

test_robust = model_df_robust.loc[
    model_df_robust["period"].eq("test")
].copy()

frozen_robust_variant_specs = {}

for name, spec in variant_specs.items():
    print("selecting robust frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val_robust,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_robust_variant_specs[name] = frozen_spec


trained_robust = {}

for name, spec in frozen_robust_variant_specs.items():
    print("robust frozen-rule training", name, "rows", len(train_val_robust))

    trained_robust[name] = train_variant(
        name,
        spec,
        train_val_robust,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

robust_rows = []
robust_frames = []

for variant, bundle in trained_robust.items():
    scored = score_variant(bundle, test_robust)  # Score the longer-window held-out test set.
    
    robust_rows.append(
        summary_row(
            variant,
            "longer_heldout_test_2025_07_to_2026_06",
            scored,
            bundle=bundle,
        )
    )
    
    robust_frames.append(
        scored.assign(
            variant=variant,
            period="longer_heldout_test_2025_07_to_2026_06",
        )
    )

robust_final_summary = pd.DataFrame(robust_rows)
robust_final_predictions = pd.concat(robust_frames, ignore_index=True)

display(
    robust_final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False
    )
)

if SAVE_OUTPUTS:
    robust_final_summary.to_csv(
        OUTPUT_DIR / "full_ltn_robust_longer_test_summary.csv",
        index=False,
    )

    robust_final_predictions.to_csv(
        OUTPUT_DIR / "full_ltn_robust_longer_test_predictions.csv",
        index=False,
    )

Robustness split counts:


target_label,bearish,bullish,neutral
period,,,
test,58,65,13
training,76,85,16
validation,32,29,6


selecting robust frozen rules full_ltn_qwen_only
selecting robust frozen rules full_ltn_context_only
selecting robust frozen rules full_ltn_qwen_plus_context
selecting robust frozen rules full_ltn_qwen_context_interaction
robust frozen-rule training full_ltn_qwen_only rows 244
robust frozen-rule training full_ltn_context_only rows 244
robust frozen-rule training full_ltn_qwen_plus_context rows 244
robust frozen-rule training full_ltn_qwen_context_interaction rows 244


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,full_ltn_qwen_only,longer_heldout_test_2025_07_to_2026_06,136,52.941176,0.78858,0.538148,0.172796,61.029412,38.970588,0.0,0.638955,True,0.3,6
3,full_ltn_qwen_context_interaction,longer_heldout_test_2025_07_to_2026_06,136,52.941176,0.78858,0.151907,0.099091,53.676471,46.323529,0.0,0.733019,True,0.3,12
2,full_ltn_qwen_plus_context,longer_heldout_test_2025_07_to_2026_06,136,52.205882,0.78858,0.345883,0.172796,63.970588,36.029412,0.0,0.739529,True,0.3,12
1,full_ltn_context_only,longer_heldout_test_2025_07_to_2026_06,136,39.705882,0.78858,-0.820686,0.927682,38.970588,61.029412,0.0,0.703688,True,0.3,1


## 11. LTN vs no LTN ablation test 1

This section compares the LTN Qwen-plus-context model against a neural classifier using the same feature set with logic disabled.


In [13]:
train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

ablation_base_spec = variant_specs["full_ltn_qwen_plus_context"]


candidate_rules, selected_rules = mine_validated_rules(
    train_val,
    ablation_base_spec["feature_cols"],
    ablation_base_spec["return_col"],
    rule_blocks=ablation_base_spec.get("rule_blocks"),
    max_pair_features=ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=ablation_base_spec.get("max_rules_per_label", 12),
)

ablation_ltn_spec = ablation_base_spec.copy()
ablation_ltn_spec["selected_rules"] = selected_rules.copy()

# display(pd.DataFrame([{
#     "variant": "full_ltn_qwen_plus_context",
#     "rule_blocks": ", ".join(ablation_base_spec.get("rule_blocks", [])),
#     "n_candidate_rules": len(candidate_rules),
#     "n_selected_rules": len(selected_rules),
#     "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
#     "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
#     "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
#     "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
# }]))


ltn_bundle = train_variant(
    "ltn_qwen_plus_context_ablation",
    ablation_ltn_spec,
    train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

ltn_scored = score_variant(ltn_bundle, test)


no_logic_bundle = train_variant(
    "no_logic_qwen_plus_context_ablation",
    ablation_base_spec,
    train_val,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

no_logic_scored = score_variant(no_logic_bundle, test)


ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "LTN, Qwen + context",
        "heldout_test",
        ltn_scored,
        bundle=ltn_bundle,
    ),
    summary_row(
        "Neural classifier, Qwen + context, no logic",
        "heldout_test",
        no_logic_scored,
        bundle=no_logic_bundle,
    ),
])

display(ltn_ablation_summary)


if SAVE_OUTPUTS:
    ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "ltn_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "no_logic_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    ltn_scored.to_csv(
        OUTPUT_DIR / "ltn_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    no_logic_scored.to_csv(
        OUTPUT_DIR / "no_logic_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "ltn_ablation_qwen_plus_context_summary.csv",
        index=False,
    )

,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,"LTN, Qwen + context",heldout_test,63,53.968254,0.337596,0.333724,0.103684,49.206349,50.793651,0.0,0.776809,True,0.3,17
1,"Neural classifier, Qwen + context, no logic",heldout_test,63,47.619048,0.337596,-0.150417,0.307328,47.619048,52.380952,0.0,0.707700,False,0.0,0


## 12. LTN vs no LTN ablation test 2

same ablation experiment as test one, but over repeated seeds


In [14]:
seed_rows = []
seed_rule_rows = []

train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

base_spec = variant_specs["full_ltn_qwen_plus_context"]

for seed in ROBUST_SEEDS:
    print(f"Main repeated-seed ablation: seed {seed}")

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    ltn_spec = base_spec.copy()
    ltn_spec["selected_rules"] = selected_rules.copy()

    seed_rule_rows.append({
        "seed": seed,
        "model": "LTN, Qwen + context",
        "rule_blocks": ", ".join(base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    # CHANGED in clean notebook:
    # LTN branch uses frozen rules, so mine_rules=False.
    ltn_bundle = train_variant(
        f"ltn_qwen_plus_context_seed_{seed}",
        ltn_spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    ltn_scored = score_variant(ltn_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "LTN, Qwen + context",
        **summary_row("LTN, Qwen + context", "heldout_test", ltn_scored, bundle=ltn_bundle),
    })

    no_logic_bundle = train_variant(
        f"no_logic_qwen_plus_context_seed_{seed}",
        base_spec,
        train_val,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    no_logic_scored = score_variant(no_logic_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "Neural classifier, Qwen + context, no logic",
        **summary_row(
            "Neural classifier, Qwen + context, no logic",
            "heldout_test",
            no_logic_scored,
            bundle=no_logic_bundle,
        ),
    })


seed_results = pd.DataFrame(seed_rows)
seed_rule_summary = pd.DataFrame(seed_rule_rows)

seed_summary = (
    seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

display(seed_rule_summary)
display(seed_results)
display(seed_summary)

Main repeated-seed ablation: seed 1
Main repeated-seed ablation: seed 2
Main repeated-seed ablation: seed 3
Main repeated-seed ablation: seed 4
Main repeated-seed ablation: seed 5
Main repeated-seed ablation: seed 7
Main repeated-seed ablation: seed 11
Main repeated-seed ablation: seed 13
Main repeated-seed ablation: seed 17
Main repeated-seed ablation: seed 19


,seed,model,rule_blocks,n_candidate_rules,n_selected_rules,validation_n_min,validation_n_median,validation_n_max,validation_accuracy_mean
0,1,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
1,2,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
2,3,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
3,4,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
4,5,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
5,7,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
6,11,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
7,13,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
8,17,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672
9,19,"LTN, Qwen + context","qwen_semantic, market_context",1203,17,5,9.0,14,68.602672


,seed,model,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,1,"LTN, Qwen + context","LTN, Qwen + context",heldout_test,63,50.793651,0.337596,0.269081,0.156752,57.142857,42.857143,0.0,0.743606,True,0.3,17
1,1,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",heldout_test,63,47.619048,0.337596,-0.053618,0.307328,50.793651,49.206349,0.0,0.730872,False,0.0,0
2,2,"LTN, Qwen + context","LTN, Qwen + context",heldout_test,63,55.555556,0.337596,0.512452,0.064959,53.968254,46.031746,0.0,0.776315,True,0.3,17
3,2,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",heldout_test,63,47.619048,0.337596,0.051691,0.307328,53.968254,46.031746,0.0,0.712653,False,0.0,0
4,3,"LTN, Qwen + context","LTN, Qwen + context",heldout_test,63,53.968254,0.337596,0.396879,0.064959,57.142857,42.857143,0.0,0.685705,True,0.3,17
5,3,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",heldout_test,63,53.968254,0.337596,0.174715,0.103684,49.206349,50.793651,0.0,0.727515,False,0.0,0
6,4,"LTN, Qwen + context","LTN, Qwen + context",heldout_test,63,50.793651,0.337596,0.383369,0.156752,53.968254,46.031746,0.0,0.691374,True,0.3,17
7,4,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",heldout_test,63,47.619048,0.337596,0.175406,0.307328,53.968254,46.031746,0.0,0.712918,False,0.0,0
8,5,"LTN, Qwen + context","LTN, Qwen + context",heldout_test,63,47.619048,0.337596,0.024643,0.400653,49.206349,50.793651,0.0,0.761361,True,0.3,17
9,5,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",heldout_test,63,57.142857,0.337596,0.940304,0.021478,53.968254,46.031746,0.0,0.730385,False,0.0,0


,model,runs,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"LTN, Qwen + context",10,52.698413,2.342428,0.384086,0.155919,0.738455,0.040296,17.0,17,17
1,"Neural classifier, Qwen + context, no logic",10,51.269841,3.513642,0.286931,0.328462,0.706822,0.025813,0.0,0,0


## 13. LTN vs no-LTN test 3 - Longer-Window Robustness

This section repeats the frozen-rule workflow under a longer held-out test window. It keeps temporal ordering as training, validation, then test.


In [15]:
robust_ablation_base_spec = variant_specs["full_ltn_qwen_plus_context"]


robust_candidate_rules, robust_selected_rules = mine_validated_rules(
    train_val_robust,
    robust_ablation_base_spec["feature_cols"],
    robust_ablation_base_spec["return_col"],
    rule_blocks=robust_ablation_base_spec.get("rule_blocks"),
    max_pair_features=robust_ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=robust_ablation_base_spec.get("max_rules_per_label", 12),
)

robust_ablation_ltn_spec = robust_ablation_base_spec.copy()
robust_ablation_ltn_spec["selected_rules"] = robust_selected_rules.copy()

# display(pd.DataFrame([{
#     "variant": "full_ltn_qwen_plus_context",
#     "rule_blocks": ", ".join(robust_ablation_base_spec.get("rule_blocks", [])),
#     "n_candidate_rules": len(robust_candidate_rules),
#     "n_selected_rules": len(robust_selected_rules),
#     "validation_n_min": robust_selected_rules["validation_n"].min() if len(robust_selected_rules) else np.nan,
#     "validation_n_median": robust_selected_rules["validation_n"].median() if len(robust_selected_rules) else np.nan,
#     "validation_n_max": robust_selected_rules["validation_n"].max() if len(robust_selected_rules) else np.nan,
#     "validation_accuracy_mean": robust_selected_rules["validation_accuracy_pct"].mean() if len(robust_selected_rules) else np.nan,
# }]))


robust_ltn_bundle = train_variant(
    "robust_ltn_qwen_plus_context_ablation",
    robust_ablation_ltn_spec,
    train_val_robust,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

robust_ltn_scored = score_variant(
    robust_ltn_bundle,
    test_robust,
)


robust_no_logic_bundle = train_variant(
    "robust_no_logic_qwen_plus_context_ablation",
    robust_ablation_base_spec,
    train_val_robust,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

robust_no_logic_scored = score_variant(
    robust_no_logic_bundle,
    test_robust,
)


robust_ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "LTN, Qwen + context",
        "longer_heldout_test_2025_07_to_2026_06",
        robust_ltn_scored,
        bundle=robust_ltn_bundle,
    ),
    summary_row(
        "Neural classifier, Qwen + context, no logic",
        "longer_heldout_test_2025_07_to_2026_06",
        robust_no_logic_scored,
        bundle=robust_no_logic_bundle,
    ),
])

display(robust_ltn_ablation_summary)


if SAVE_OUTPUTS:
    robust_ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "robust_ltn_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    robust_no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "robust_no_logic_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    robust_ltn_scored.to_csv(
        OUTPUT_DIR / "robust_ltn_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    robust_no_logic_scored.to_csv(
        OUTPUT_DIR / "robust_no_logic_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    robust_ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "robust_ltn_ablation_qwen_plus_context_summary.csv",
        index=False,
    )

,variant,rule_blocks,n_candidate_rules,n_selected_rules,validation_n_min,validation_n_median,validation_n_max,validation_accuracy_mean
0,full_ltn_qwen_plus_context,"qwen_semantic, market_context",906,12,5,6.0,15,70.049603


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,"LTN, Qwen + context",longer_heldout_test_2025_07_to_2026_06,136,50.735294,0.78858,0.190693,0.334133,63.235294,36.764706,0.0,0.737155,True,0.3,12
1,"Neural classifier, Qwen + context, no logic",longer_heldout_test_2025_07_to_2026_06,136,51.470588,0.78858,0.180551,0.274251,53.676471,46.323529,0.0,0.665103,False,0.0,0


## 14. LTN vs No LTN test 4 - longer window robust set and repeated seeds

This section repeats the main and longer-window LTN/no-logic comparisons across multiple random seeds.


In [16]:
robust_seed_rows = []
robust_seed_rule_rows = []
robust_seed_scored_frames = []

robust_base_spec = variant_specs["full_ltn_qwen_plus_context"]

for seed in ROBUST_SEEDS:
    print(f"Longer-window repeated seed: {seed}")

    robust_candidate_rules, robust_selected_rules = mine_validated_rules(
        train_val_robust,
        robust_base_spec["feature_cols"],
        robust_base_spec["return_col"],
        rule_blocks=robust_base_spec.get("rule_blocks"),
        max_pair_features=robust_base_spec.get("max_pair_features", 36),
        max_rules_per_label=robust_base_spec.get("max_rules_per_label", 12),
    )

    robust_seed_rule_rows.append({
        "seed": seed,
        "rule_blocks": ", ".join(robust_base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(robust_candidate_rules),
        "n_selected_rules": len(robust_selected_rules),
    })

    robust_ltn_spec = robust_base_spec.copy()
    robust_ltn_spec["selected_rules"] = robust_selected_rules.copy()

    robust_ltn_bundle = train_variant(
        f"robust_ltn_qwen_plus_context_seed_{seed}",
        robust_ltn_spec,
        train_val_robust,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    robust_ltn_scored = score_variant(robust_ltn_bundle, test_robust)

    robust_seed_scored_frames.append(
        robust_ltn_scored.assign(
            seed=seed,
            model="LTN, Qwen + context",
        )
    )

    robust_seed_rows.append({
        "seed": seed,
        "model": "LTN, Qwen + context",
        **summary_row(
            "LTN, Qwen + context",
            "longer_heldout_test",
            robust_ltn_scored,
            bundle=robust_ltn_bundle,
        ),
    })

    robust_no_logic_bundle = train_variant(
        f"robust_no_logic_qwen_plus_context_seed_{seed}",
        robust_base_spec,
        train_val_robust,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    robust_no_logic_scored = score_variant(robust_no_logic_bundle, test_robust)

    robust_seed_scored_frames.append(
        robust_no_logic_scored.assign(
            seed=seed,
            model="Neural classifier, Qwen + context, no logic",
        )
    )

    robust_seed_rows.append({
        "seed": seed,
        "model": "Neural classifier, Qwen + context, no logic",
        **summary_row(
            "Neural classifier, Qwen + context, no logic",
            "longer_heldout_test",
            robust_no_logic_scored,
            bundle=robust_no_logic_bundle,
        ),
    })

robust_seed_results = pd.DataFrame(robust_seed_rows)
robust_seed_rule_summary = pd.DataFrame(robust_seed_rule_rows)

robust_seed_predictions = pd.concat(robust_seed_scored_frames, ignore_index=True)

robust_seed_summary = (
    robust_seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        events_mean=("n", "mean"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        signflip_p_mean=("aligned_signflip_p", "mean"),
        bullish_rate_mean=("bullish_pred_rate_pct", "mean"),
        bearish_rate_mean=("bearish_pred_rate_pct", "mean"),
        neutral_rate_mean=("neutral_pred_rate_pct", "mean"),
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

robust_confusion_tables = {}

for model_name, group in robust_seed_predictions.groupby("model"):
    cm_counts = pd.crosstab(
        group["target_label"],
        group["prediction"],
        rownames=["True label"],
        colnames=["Predicted label"],
        dropna=False,
    ).reindex(index=LABELS, columns=LABELS, fill_value=0)

    cm_row_pct = (
        cm_counts
        .div(cm_counts.sum(axis=1).replace(0, np.nan), axis=0)
        .mul(100)
        .round(1)
    )

    robust_confusion_tables[model_name] = {
        "counts": cm_counts,
        "row_pct": cm_row_pct,
    }

display(robust_seed_rule_summary)
display(robust_seed_results)
display(robust_seed_summary)

for model_name, tables in robust_confusion_tables.items():
    print(model_name)
    display(tables["counts"])
    display(tables["row_pct"])

Longer-window repeated seed: 1
Longer-window repeated seed: 2
Longer-window repeated seed: 3
Longer-window repeated seed: 4
Longer-window repeated seed: 5
Longer-window repeated seed: 7
Longer-window repeated seed: 11
Longer-window repeated seed: 13
Longer-window repeated seed: 17
Longer-window repeated seed: 19


,seed,rule_blocks,n_candidate_rules,n_selected_rules
0,1,"qwen_semantic, market_context",906,12
1,2,"qwen_semantic, market_context",906,12
2,3,"qwen_semantic, market_context",906,12
3,4,"qwen_semantic, market_context",906,12
4,5,"qwen_semantic, market_context",906,12
5,7,"qwen_semantic, market_context",906,12
6,11,"qwen_semantic, market_context",906,12
7,13,"qwen_semantic, market_context",906,12
8,17,"qwen_semantic, market_context",906,12
9,19,"qwen_semantic, market_context",906,12


,seed,model,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,1,"LTN, Qwen + context","LTN, Qwen + context",longer_heldout_test,136,50.000000,0.78858,0.224059,0.334133,58.823529,41.176471,0.0,0.713180,True,0.3,12
1,1,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",longer_heldout_test,136,50.000000,0.78858,0.249534,0.334133,55.882353,44.117647,0.0,0.714667,False,0.0,0
2,2,"LTN, Qwen + context","LTN, Qwen + context",longer_heldout_test,136,50.735294,0.78858,0.310995,0.274251,63.970588,36.029412,0.0,0.710068,True,0.3,12
3,2,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",longer_heldout_test,136,49.264706,0.78858,0.122795,0.334133,55.882353,44.117647,0.0,0.673526,False,0.0,0
4,3,"LTN, Qwen + context","LTN, Qwen + context",longer_heldout_test,136,46.323529,0.78858,-0.097651,0.665867,55.147059,44.852941,0.0,0.748519,True,0.3,12
5,3,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",longer_heldout_test,136,54.411765,0.78858,0.437115,0.051457,59.558824,40.441176,0.0,0.705890,False,0.0,0
6,4,"LTN, Qwen + context","LTN, Qwen + context",longer_heldout_test,136,55.147059,0.78858,0.398381,0.051457,58.088235,41.911765,0.0,0.785806,True,0.3,12
7,4,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",longer_heldout_test,136,53.676471,0.78858,0.356365,0.132446,58.823529,41.176471,0.0,0.696501,False,0.0,0
8,5,"LTN, Qwen + context","LTN, Qwen + context",longer_heldout_test,136,53.676471,0.78858,0.384733,0.132446,57.352941,42.647059,0.0,0.712959,True,0.3,12
9,5,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",longer_heldout_test,136,49.264706,0.78858,0.367121,0.398551,63.970588,36.029412,0.0,0.734149,False,0.0,0


,model,runs,events_mean,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,signflip_p_mean,bullish_rate_mean,bearish_rate_mean,neutral_rate_mean,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"LTN, Qwen + context",10,136.0,50.955882,2.663574,0.212671,0.155486,0.305355,59.191176,40.808824,0.0,0.720558,0.034708,12.0,12,12
1,"Neural classifier, Qwen + context, no logic",10,136.0,51.250000,2.326496,0.234174,0.218696,0.261150,56.397059,43.602941,0.0,0.686863,0.028598,0.0,0,0


LTN, Qwen + context


Predicted label,bearish,bullish,neutral
True label,,,
bearish,288,292,0
bullish,245,405,0
neutral,22,108,0


Predicted label,bearish,bullish,neutral
True label,,,
bearish,49.7,50.3,0.0
bullish,37.7,62.3,0.0
neutral,16.9,83.1,0.0


Neural classifier, Qwen + context, no logic


Predicted label,bearish,bullish,neutral
True label,,,
bearish,306,274,0
bullish,259,391,0
neutral,28,102,0


Predicted label,bearish,bullish,neutral
True label,,,
bearish,52.8,47.2,0.0
bullish,39.8,60.2,0.0
neutral,21.5,78.5,0.0
